# Car Price Prediction with Machine Learning

**OASIS INFOBYTE - Data Science Task 3**  
Predict the selling price of a used car from its brand, age, mileage, fuel type, seller type, transmission, ownership history, and current showroom price.

**Dataset:** [Vehicle Dataset from CarDekho on Kaggle](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho), accessed through a public CSV mirror. Selling and showroom prices are in Indian lakh rupees.

## 1. Setup and data loading

The notebook defaults to a public mirror so it can be run without Kaggle credentials. To use a downloaded Kaggle copy instead, set `DATA_PATH` to its CSV path.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
RANDOM_STATE = 42
CURRENT_YEAR = 2026
DATA_PATH = Path('car_data.csv')  # Included dataset snapshot

source = DATA_PATH
cars = pd.read_csv(source)
cars.head()

## 2. Data quality report

We inspect dimensions, types, missingness, duplicate records, and numeric ranges before making changes.

In [ ]:
print(f'Shape: {cars.shape[0]:,} rows x {cars.shape[1]} columns')
display(cars.dtypes.to_frame('dtype'))
quality_report = pd.DataFrame({
    'missing_values': cars.isna().sum(),
    'missing_pct': (cars.isna().mean() * 100).round(2),
    'unique_values': cars.nunique()
}).sort_values('missing_values', ascending=False)
display(quality_report)
print(f'Duplicate rows: {cars.duplicated().sum():,}')
display(cars.describe(include='all').T)

## 3. Cleaning and feature engineering

Column names are standardized, text values are trimmed and title-cased, exact duplicates are removed, and invalid numeric records are excluded. Missing numeric predictors are imputed later inside the modelling pipeline using the median; missing categoricals use the most frequent value. This avoids leakage from the test set.

`car_age` is calculated from manufacturing year, and `brand` is the first token in the car name. The raw name is then removed because it is highly granular and would create sparse categories in this small dataset.

In [ ]:
df = cars.copy()
df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_'))

text_columns = df.select_dtypes(include='object').columns
for column in text_columns:
    df[column] = df[column].astype('string').str.strip().str.title()

before_duplicates = len(df)
df = df.drop_duplicates().copy()
removed_duplicates = before_duplicates - len(df)

numeric_columns = ['year', 'selling_price', 'present_price', 'kms_driven', 'owner']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Values outside these bounds are not plausible for this dataset and are excluded.
valid_rows = (
    df['year'].between(1980, CURRENT_YEAR)
    & df['selling_price'].gt(0)
    & df['present_price'].gt(0)
    & df['kms_driven'].ge(0)
    & df['owner'].ge(0)
)
df = df.loc[valid_rows].copy()

# The supplied file also contains two-wheelers; remove them for a car-only model.
two_wheeler_pattern = (
    r'Royal Enfield|Ktm |Bajaj |Hyosung|Yamaha |Tvs |Hero |Um |'
    r'Honda (Cb|Activa|Dream)|Suzuki Access|Mahindra Mojo|Activa '
)
two_wheeler_rows = df['car_name'].str.contains(two_wheeler_pattern, case=False, na=False, regex=True)
df = df.loc[~two_wheeler_rows].copy()
df['car_age'] = CURRENT_YEAR - df['year']
df['brand'] = df['car_name'].str.split().str[0].str.title()

print(f'Exact duplicates removed: {removed_duplicates}')
print(f'Rows retained after validation: {len(df):,}')
display(df.head())
display(df[['fuel_type', 'seller_type', 'transmission', 'brand']].nunique().to_frame('unique_values'))

## 4. Exploratory data analysis

The following charts examine the target distribution and the relationships requested in the task. Outlying prices are retained: they may be valid premium vehicles, and removing them without business context can bias a pricing model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
sns.histplot(data=df, x='selling_price', bins=30, kde=True, ax=axes[0], color='#2A6F97')
axes[0].set(title='Selling Price Distribution', xlabel='Selling price (lakh INR)')
sns.boxplot(data=df, x='fuel_type', y='selling_price', ax=axes[1], hue='fuel_type', legend=False)
axes[1].set(title='Selling Price by Fuel Type', xlabel='Fuel type', ylabel='Selling price (lakh INR)')
sns.scatterplot(data=df, x='car_age', y='selling_price', hue='transmission', alpha=0.75, ax=axes[2])
axes[2].set(title='Selling Price vs Car Age', xlabel='Car age (years)', ylabel='Selling price (lakh INR)')
plt.tight_layout()
plt.show()

numeric_for_heatmap = ['selling_price', 'present_price', 'kms_driven', 'owner', 'car_age']
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_for_heatmap].corr(), annot=True, cmap='vlag', center=0, fmt='.2f')
plt.title('Numeric Feature Correlations')
plt.show()

**EDA interpretation:** Selling price is expected to be right-skewed because a few expensive cars raise the upper tail. Compare fuel-type medians and spread rather than relying only on individual points. The age scatter helps assess depreciation, while the correlation heatmap identifies numeric features associated with price; correlation alone does not establish causation.

## 5. Modelling

The target is `selling_price`. Preprocessing is inside each pipeline so the training-set imputation and one-hot encoding are not fitted using holdout data. We compare a linear baseline against a non-linear Random Forest model.

In [ ]:
features = ['present_price', 'kms_driven', 'owner', 'car_age', 'fuel_type',
            'seller_type', 'transmission', 'brand']
target = 'selling_price'
X = df[features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_features = ['present_price', 'kms_driven', 'owner', 'car_age']
categorical_features = ['fuel_type', 'seller_type', 'transmission', 'brand']
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=500, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    )
}

fitted_models, scores = {}, []
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    fitted_models[name] = pipeline
    scores.append({
        'model': name,
        'MAE': mean_absolute_error(y_test, predictions),
        'RMSE': mean_squared_error(y_test, predictions) ** 0.5,
        'R2': r2_score(y_test, predictions)
    })

results = pd.DataFrame(scores).sort_values('RMSE').reset_index(drop=True)
display(results.style.format({'MAE': '{:.3f}', 'RMSE': '{:.3f}', 'R2': '{:.3f}'}))
best_name = results.loc[0, 'model']
best_model = fitted_models[best_name]
print(f'Best model by lowest RMSE: {best_name}')

## 6. Best-model diagnostics and feature importance

MAE is the typical error in lakh INR, RMSE penalizes larger errors more heavily, and R² is the share of target variation explained on the unseen test set. The feature-importance plot below is available when Random Forest is selected as best.

In [ ]:
best_predictions = best_model.predict(X_test)
plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=best_predictions, alpha=0.75)
limits = [min(y_test.min(), best_predictions.min()), max(y_test.max(), best_predictions.max())]
plt.plot(limits, limits, '--', color='crimson', label='Perfect prediction')
plt.xlabel('Actual selling price (lakh INR)')
plt.ylabel('Predicted selling price (lakh INR)')
plt.title(f'Actual vs Predicted - {best_name}')
plt.legend()
plt.tight_layout()
plt.show()

if best_name == 'Random Forest':
    transformed_names = best_model.named_steps['preprocessor'].get_feature_names_out()
    importances = pd.Series(
        best_model.named_steps['model'].feature_importances_, index=transformed_names
    ).sort_values(ascending=False).head(15).sort_values()
    plt.figure(figsize=(9, 7))
    importances.plot(kind='barh', color='#2A6F97')
    plt.title('Top 15 Random Forest Feature Importances')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    print('Random Forest was not the best model. Fit it separately to inspect tree-based feature importances.')

## 7. Executed results and conclusion

The supplied data had **301 records**. After removing **2 exact duplicates** and **99 two-wheeler listings**, the car-only analysis used **200 records** (160 training, 40 test). There were no missing values after cleaning.

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Linear Regression | 1.171 | 1.814 | 0.809 |
| Random Forest | 0.976 | 1.479 | 0.873 |

**Recommendation:** Random Forest is the preferred model because it achieved the lower MAE and RMSE and the higher R² on the same unseen test set. Its MAE is approximately **0.98 lakh INR**. Use the diagnostics and feature-importance plots above to explain the prediction behavior. Important limitations: the small dataset may not represent all regions or model years, and `Present_Price` can dominate predictions because it is closely related to resale value.